# dfir-llm-assistant sur GPU gratuit (Kaggle ou Google Colab)

Ce notebook lance toute la chaîne du projet sur une machine distante avec GPU :
**Ollama** (le moteur de modèles) → **LiteLLM** (la passerelle) → **dfa** (le client), puis mesure la qualité avec `dfa eval`.

⚠️ **Règle de confidentialité** : Kaggle et Colab sont des clouds Google. N'utiliser ici **que les données fictives ou publiques** du dépôt (`samples/`, EVTX-ATTACK-SAMPLES). Jamais de vrais journaux d'enquête.

**Avant de lancer :**
- **Kaggle** : *Settings → Accelerator → GPU T4 x2* et *Settings → Internet → On* (compte vérifié par téléphone).
- **Colab** : *Exécution → Modifier le type d'exécution → GPU T4*.

Puis *Run all*. Prévoir 10 à 15 minutes la première fois (téléchargement du modèle).

## 1. Choisir le modèle

| Modèle | Taille | Tient sur | Remarque |
|---|---|---|---|
| `qwen2.5:7b-instruct` | 4,7 Go | 1 T4 | rapide, le défaut du projet |
| `qwen2.5:14b-instruct` | 9 Go | 1 T4 | **recommandé** : bien meilleur en analyse |
| `qwen2.5:32b-instruct` | 20 Go | 2 T4 (Kaggle) | le plus pertinent, plus lent |

`CONTEXT` est la fenêtre de contexte : combien de tokens le modèle lit à la fois. La valeur par défaut d'Ollama (4 096) tronque les prompts de `dfa` : elle est portée ici à 16 384.

In [ ]:
MODEL = "qwen2.5:14b-instruct"   # change ici pour comparer
CONTEXT = 16384
SMALL_MODEL = "qwen2.5:7b-instruct"  # second modèle, pour comparer dans l'évaluation (mettre None pour l'ignorer)

## 2. Vérifier le GPU
`nvidia-smi` affiche la ou les cartes graphiques disponibles. Si la commande échoue, le GPU n'est pas activé (voir les réglages en haut).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Installer et lancer Ollama
Pas de Docker sur Kaggle ni Colab : on installe Ollama directement avec son script officiel, puis on lance le serveur **en arrière-plan** (`subprocess.Popen`), pour que le notebook reste utilisable.
`OLLAMA_CONTEXT_LENGTH` règle la taille de contexte par défaut de tous les modèles.

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama-install.log 2>&1; tail -n 2 /tmp/ollama-install.log

import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH=str(CONTEXT), OLLAMA_KEEP_ALIVE="60m", OLLAMA_HOST="127.0.0.1:11434")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    print(open("/tmp/ollama.log").read()[-2000:])
    raise RuntimeError("Ollama ne répond pas : voir le journal ci-dessus")

## 4. Télécharger le(s) modèle(s)
Ici le serveur Ollama a accès à Internet, alors que dans le docker-compose, le serveur qui sert les modèles n'en a pas.

In [ ]:
!ollama pull {MODEL}
if SMALL_MODEL:
    !ollama pull {SMALL_MODEL}
!ollama list

## 5. Récupérer le projet et installer le client `dfa`
`evtx` (bibliothèque Python) sert à convertir les fichiers `.evtx` publics, à la place de l'outil `evtx_dump` de nix-forensics.

In [ ]:
import pathlib
WORK = "/kaggle/working" if pathlib.Path("/kaggle/working").exists() else "/content"
REPO = f"{WORK}/dfir-llm-assistant"
!git clone -q https://github.com/daren-houedanou/dfir-llm-assistant {REPO} 2>/dev/null || git -C {REPO} pull -q
%cd {REPO}
!pip install -q -e . "litellm[proxy]" evtx

## 6. Configurer et lancer la passerelle LiteLLM
On garde **la même architecture qu'en production** : `dfa` parle à LiteLLM, qui route vers Ollama.
On écrit une config où l'alias `dfir-local` pointe sur le gros modèle et `dfir-local-small` sur le petit. Le code de `dfa` ne change pas : c'est tout l'intérêt de la passerelle.

In [ ]:
import secrets, textwrap, pathlib
KEY = "sk-" + secrets.token_hex(16)   # clé maître générée pour cette session
small = SMALL_MODEL or MODEL
config = textwrap.dedent(f'''
model_list:
  - model_name: dfir-local
    litellm_params: {{model: "ollama_chat/{MODEL}", api_base: "http://127.0.0.1:11434"}}
  - model_name: dfir-local-small
    litellm_params: {{model: "ollama_chat/{small}", api_base: "http://127.0.0.1:11434"}}
router_settings:
  num_retries: 0
  timeout: 600
litellm_settings:
  drop_params: true
  request_timeout: 600
  turn_off_message_logging: true
general_settings:
  master_key: {KEY}
''')
pathlib.Path("/tmp/litellm.yaml").write_text(config)

# If the notebook is re-run in the same session, stop the previous gateway: it still holds port 4000
# with the OLD key, and the new key would be rejected ("No connected db").
subprocess.run(["pkill", "-f", "litellm --config"]); time.sleep(3)
litellm_proc = subprocess.Popen(["litellm", "--config", "/tmp/litellm.yaml", "--port", "4000"],
                                stdout=open("/tmp/litellm.log", "w"), stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        urllib.request.urlopen("http://127.0.0.1:4000/health/readiness"); print("LiteLLM prêt"); break
    except Exception:
        time.sleep(2)
else:
    print(open("/tmp/litellm.log").read()[-3000:])
    raise RuntimeError("LiteLLM ne répond pas : voir le journal ci-dessus")

os.environ.update(DFA_GATEWAY_URL="http://127.0.0.1:4000", DFA_API_KEY=KEY, DFA_TIMEOUT="600",
                  DFA_AUDIT_LOG="/tmp/dfa-audit.jsonl")

## 7. Premiers essais
Les mêmes commandes qu'en local. La ligne `latency=` indique le temps de réponse.
Avec 16 384 tokens de contexte, on peut garder le budget par défaut (6 000).

In [ ]:
!dfa models
!dfa triage samples/win-rdp-intrusion

In [ ]:
!dfa summarize samples/win-rdp-intrusion

In [ ]:
!dfa ask samples/linux-ssh-compromise -q "Quel compte a été compromis et comment ? Cite les preuves."

In [ ]:
!dfa query "connexions RDP depuis une IP externe suivies d'une création de compte" -t sigma

## 8. Évaluation : règles seules vs LLM
`dfa eval` compare automatiquement la référence « règles seules » et chaque modèle sur les scénarios annotés : verdict, précision et rappel ATT&CK, IOC, questions/réponses, hallucinations, latence et fuites de données.

**Jeu synthétique** (4 scénarios, dont une journée normale pour mesurer les fausses alertes) :

In [ ]:
MODELS = "-m dfir-local" + (" -m dfir-local-small" if SMALL_MODEL else "")
!dfa eval eval/scenarios {MODELS} -r /tmp/eval-synthetic.md
from IPython.display import Markdown, display
display(Markdown(open("/tmp/eval-synthetic.md").read()))

**Jeu public** (8 vrais enregistrements d'attaques, EVTX-ATTACK-SAMPLES) , utilisés pour mettre au point les règles :

In [ ]:
!bash scripts/fetch_public_datasets.sh
!dfa eval eval/public {MODELS} -r /tmp/eval-public.md
display(Markdown(open("/tmp/eval-public.md").read()))

**Jeu « jamais vu »** (`eval/holdout`, 10 autres enregistrements EVTX-ATTACK-SAMPLES) : les règles n'ont **pas** été écrites en les regardant. C'est le test qui dit si les résultats généralisent.


In [ ]:
!dfa eval eval/holdout {MODELS} -r /tmp/eval-holdout.md
display(Markdown(open("/tmp/eval-holdout.md").read()))


**Image mémoire réelle** (`eval/memory`) : un poste Windows XP infecté par le cheval de Troie bancaire Cridex. Le script télécharge l'image, lance Volatility 3 (`pslist`, `cmdline`, `netscan`, `malfind`) et écrit les sorties JSON dans `data/memory/cridex`.

Le site d'origine ne répond plus : le script essaie ensuite la copie de la Wayback Machine. Pour une autre image, utiliser `MEMORY_URL` (lien vers un `.zip`, `.raw`, `.vmem` ou `.mem`) ou `MEMORY_FILE` (fichier déjà présent, par exemple un jeu de données Kaggle ajouté avec « Add Input »).

In [ ]:
!bash scripts/fetch_memory_sample.sh && dfa triage data/memory/cridex

In [ ]:
!dfa eval eval/memory {MODELS} -r /tmp/eval-memory.md
display(Markdown(open("/tmp/eval-memory.md").read()))

## 9. Reproductibilité : rejouer les appels
Chaque appel au modèle a été journalisé (`/tmp/dfa-audit.jsonl`). On vérifie l'intégrité du journal, puis on rejoue quelques requêtes pour voir si les réponses sont identiques octet pour octet (température 0 et seed fixe).

In [ ]:
!dfa audit verify /tmp/dfa-audit.jsonl --skip-inputs
!dfa audit replay /tmp/dfa-audit.jsonl -n 0
!dfa audit replay /tmp/dfa-audit.jsonl -n 1

## 10. Récupérer les résultats
Les rapports sont copiés dans le dossier de sortie (`/kaggle/working` sur Kaggle, `/content` sur Colab).

In [ ]:
import shutil
out = pathlib.Path(WORK)
for f in ["/tmp/eval-synthetic.md", "/tmp/eval-public.md", "/tmp/eval-holdout.md", "/tmp/eval-memory.md", "/tmp/dfa-audit.jsonl"]:
    if pathlib.Path(f).exists():
        shutil.copy(f, out)
        print("copié :", f, "->", out)
    else:
        print("MANQUANT :", f, "(une étape précédente a échoué, regarde sa sortie)")